In [434]:
import nflreadpy as nfl
import pandas as pd
import numpy as np

In [435]:
SEASON = 2025
pbp = nfl.load_pbp([SEASON]).to_pandas()
schedule = nfl.load_schedules([SEASON]).to_pandas()

In [436]:
scrimmage_plays = pbp.loc[
    pbp["posteam"].notna()
    & pbp["defteam"].notna()
    & pbp["epa"].notna()
    & pbp["success"].notna()
    & (pbp["pass"].eq(1) | pbp["rush"].eq(1))
    & pbp["qb_kneel"].fillna(0).ne(1)
    & pbp["qb_spike"].fillna(0).ne(1)
    & pbp["two_point_attempt"].fillna(0).ne(1)
].copy()

In [437]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
)

In [438]:
pass_df = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
)
rush_df = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
)

In [439]:
offense_df = (
    offense
    .merge(pass_df, on=['game_id', 'posteam'], how='left')
    .merge(rush_df, on=['game_id', 'posteam'], how='left')
    .rename(columns={'posteam': 'team'})
)
offense_df.head()

,game_id,team,off_epa,off_success_rate,pass_epa,rush_epa
0,2025_01_ARI_NO,ARI,0.061773,0.424242,0.131530,-0.052630
1,2025_01_ARI_NO,NO,-0.057140,0.424658,-0.014034,-0.157067
2,2025_01_BAL_BUF,BAL,0.394435,0.509804,0.530099,0.282997
3,2025_01_BAL_BUF,BUF,0.302612,0.512821,0.457560,-0.046021
4,2025_01_CAR_JAX,CAR,-0.176198,0.365079,-0.214952,-0.092878


In [440]:
defense_df = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)
defense_df.head()

,game_id,team,def_epa_allowed,def_success_rate_allowed
0,2025_01_ARI_NO,ARI,-0.057140,0.424658
1,2025_01_ARI_NO,NO,0.061773,0.424242
2,2025_01_BAL_BUF,BAL,0.302612,0.512821
3,2025_01_BAL_BUF,BUF,0.394435,0.509804
4,2025_01_CAR_JAX,CAR,0.103053,0.460317


In [441]:
schedule_df = schedule.loc[
    schedule['home_score'].notna()
    & schedule['away_score'].notna(),
    ['game_id', 'season', 'week', 'gameday', 'game_type', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rest', 'away_rest', 'location'],
].copy()

In [442]:
schedule_df['home_win'] = np.where(
    schedule_df['home_score'].eq(schedule_df['away_score']), np.nan,
    schedule_df['home_score'].gt(schedule_df['away_score']).astype(int)
)

In [443]:
schedule_df['neutral_site'] = (schedule_df['location'].astype(str).str.lower().eq('neutral')).astype(int)

In [444]:
home_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'game_type', 'neutral_site']].copy()

home_rows['team'] = schedule_df['home_team']
home_rows['opponent'] = schedule_df['away_team']
home_rows['points_scored'] = schedule_df['home_score']
home_rows['points_allowed'] = schedule_df['away_score']
home_rows['rest_days'] = schedule_df['home_rest']
home_rows['is_home'] = 1

away_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'game_type', 'neutral_site']].copy()

away_rows['team'] = schedule_df['away_team']
away_rows['opponent'] = schedule_df['home_team']
away_rows['points_scored'] = schedule_df['away_score']
away_rows['points_allowed'] = schedule_df['home_score']
away_rows['rest_days'] = schedule_df['away_rest']
away_rows['is_home'] = 0

In [445]:
def add_team_result(df):
    df = df.copy()

    df['win'] = np.select(
        [
            df['points_scored'].gt(df['points_allowed']),
            df['points_scored'].lt(df['points_allowed'])
        ],
        [1.0, 0.0],
        default=0.5,
    )
    df['home_field'] = (
    df['is_home'].eq(1) 
    & df['neutral_site'].eq(0)
    ).astype(int)

    return df

home_rows = add_team_result(home_rows)
away_rows = add_team_result(away_rows)

In [446]:
schedule_team_games_df = pd.concat([home_rows, away_rows], ignore_index=True)

In [447]:
team_games_df = (
    schedule_team_games_df
    .merge(
        offense_df, 
        on=['game_id', 'team'], 
        how='left',
    )
    .merge(
        defense_df, 
        on=['game_id', 'team'], 
        how='left',
    )    
)

In [448]:
team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [449]:
team_games_df = team_games_df[[
    'season',
    'week',
    'gameday',  
    'game_id',
    'game_type',
    'team',
    'opponent',
    'is_home',
    'neutral_site',
    'home_field',
    'rest_days',
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
    'points_scored',
    'points_allowed',
    'win',
]]

In [450]:
team_games_df.loc[
    team_games_df["game_id"] == "2025_01_BAL_BUF"
]

,season,week,gameday,game_id,game_type,team,opponent,is_home,neutral_site,home_field,rest_days,off_epa,off_success_rate,pass_epa,rush_epa,def_epa_allowed,def_success_rate_allowed,points_scored,points_allowed,win
34,2025,1,2025-09-07,2025_01_BAL_BUF,REG,BAL,BUF,0,0,0,7,0.394435,0.509804,0.530099,0.282997,0.302612,0.512821,40,41,0.0
51,2025,1,2025-09-07,2025_01_BAL_BUF,REG,BUF,BAL,1,0,1,7,0.302612,0.512821,0.457560,-0.046021,0.394435,0.509804,41,40,1.0


In [451]:
team_games_df['pregame_off_epa'] = (
    team_games_df
    .groupby(['season', 'team'])['off_epa']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [452]:
team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [453]:
pregame_feature_map = {
    'off_epa': 'pregame_off_epa',
    'off_success_rate': 'pregame_off_success_rate',
    'pass_epa': 'pregame_pass_epa',
    'rush_epa': 'pregame_rush_epa',
    'def_epa_allowed': 'pregame_def_epa_allowed',
    'def_success_rate_allowed': 'pregame_def_success_rate_allowed',
    'points_scored': 'pregame_ppg',
    'points_allowed': 'pregame_ppg_allowed',
    'win': 'pregame_win_pct',
}


In [454]:
for raw_column, pregame_column in pregame_feature_map.items():
    team_games_df[pregame_column] = (
        team_games_df
        .groupby(['season', 'team'], sort=False)[raw_column]
        .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
    )

In [455]:
team_games_df['prior_games'] = (
    team_games_df
    .groupby(['season', 'team'], sort=False)
    .cumcount()
)

In [456]:
team_games_df.loc[
    (team_games_df['season'].eq(SEASON))
    & (team_games_df['team'].eq('NE')),
    ['week', 'gameday', 'opponent', 'off_epa', 'pregame_off_epa', 'points_scored', 'pregame_ppg', 'win', 'pregame_win_pct','prior_games']
]

,week,gameday,opponent,off_epa,pregame_off_epa,points_scored,pregame_ppg,win,pregame_win_pct,prior_games
372,1,2025-09-07,LV,-0.045775,NaN,13,NaN,0.0,NaN,0
373,2,2025-09-14,MIA,0.275018,-0.045775,33,13.000000,1.0,0.000000,1
374,3,2025-09-21,PIT,-0.109990,0.114622,14,23.000000,0.0,0.500000,2
375,4,2025-09-28,CAR,0.354956,0.039751,42,20.000000,1.0,0.333333,3
376,5,2025-10-05,BUF,0.065326,0.118552,23,25.500000,1.0,0.500000,4
377,6,2025-10-12,NO,0.138688,0.107907,25,25.000000,1.0,0.600000,5
378,7,2025-10-19,TEN,0.267477,0.113037,31,25.000000,1.0,0.666667,6
379,8,2025-10-26,CLE,0.093407,0.135100,32,25.857143,1.0,0.714286,7
380,9,2025-11-02,ATL,0.013419,0.129888,24,26.625000,1.0,0.750000,8
381,10,2025-11-09,TB,0.193619,0.116947,28,26.333333,1.0,0.777778,9


In [457]:
pregame_columns = list(pregame_feature_map.values())

first_team_games = (
    team_games_df
    .groupby(["season", "team"], sort=False)
    .head(1)
)

assert first_team_games[pregame_columns].isna().all().all()

In [458]:
team_feature_columns = [
    'game_id', 'team', 'prior_games', *pregame_columns,
]
pregame_team_features_df = team_games_df[team_feature_columns].copy()

In [459]:
home_features_df = pregame_team_features_df.rename(
    columns={
        'team': 'home_team',
        'prior_games': 'home_prior_games',
        **{
            column: f"home_{column}"
            for column in pregame_columns
        },
    }
)

In [460]:
away_features_df = pregame_team_features_df.rename(
    columns={
        'team': 'away_team',
        'prior_games': 'away_prior_games',
        **{
            column: f"away_{column}"
            for column in pregame_columns
        },
    }
)

In [461]:
game_columns = [
    'game_id', 'season', 'week', 'gameday', 'game_type', 'home_team', 'away_team', 'neutral_site', 'home_rest', 'away_rest', 'home_win'
]
games = schedule_df[game_columns].copy()

In [462]:
games = games.rename(
    columns={
        'home_rest': 'home_rest_days',
        'away_rest': 'away_rest_days',
    }
)
games['home_field'] = 1 - games['neutral_site']

In [463]:
model_df = (
    games
    .merge(home_features_df, on=['game_id', 'home_team'], how='left')
    .merge(away_features_df, on=['game_id', 'away_team'], how='left')
)
model_df.head()

,game_id,season,week,gameday,game_type,home_team,away_team,neutral_site,home_rest_days,away_rest_days,...,away_prior_games,away_pregame_off_epa,away_pregame_off_success_rate,away_pregame_pass_epa,away_pregame_rush_epa,away_pregame_def_epa_allowed,away_pregame_def_success_rate_allowed,away_pregame_ppg,away_pregame_ppg_allowed,away_pregame_win_pct
0,2025_01_DAL_PHI,2025,1,2025-09-04,REG,PHI,DAL,0,7,7,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025_01_KC_LAC,2025,1,2025-09-05,REG,LAC,KC,1,7,7,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025_01_TB_ATL,2025,1,2025-09-07,REG,ATL,TB,0,7,7,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025_01_CIN_CLE,2025,1,2025-09-07,REG,CLE,CIN,0,7,7,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025_01_MIA_IND,2025,1,2025-09-07,REG,IND,MIA,0,7,7,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [464]:
model_df = model_df.loc[
    model_df['home_prior_games'].ge(1)
    & model_df['away_prior_games'].ge(1)
].copy()

In [465]:
model_df = model_df.dropna(subset=['home_win']).copy()
model_df['home_win'] = model_df['home_win'].astype(int)

In [466]:
model_df = model_df.drop(columns=['home_prior_games', 'away_prior_games'])

In [467]:
model_df = model_df.sort_values(['gameday', 'game_id']).reset_index(drop=True)

In [468]:
model_df.head()

,game_id,season,week,gameday,game_type,home_team,away_team,neutral_site,home_rest_days,away_rest_days,...,home_pregame_win_pct,away_pregame_off_epa,away_pregame_off_success_rate,away_pregame_pass_epa,away_pregame_rush_epa,away_pregame_def_epa_allowed,away_pregame_def_success_rate_allowed,away_pregame_ppg,away_pregame_ppg_allowed,away_pregame_win_pct
0,2025_02_WAS_GB,2025,2,2025-09-11,REG,GB,WAS,0,4,4,...,1.0,0.155313,0.522388,0.098424,0.245021,-0.138282,0.367647,21.0,6.0,1.0
1,2025_02_ATL_MIN,2025,2,2025-09-14,REG,MIN,ATL,0,6,7,...,1.0,0.075180,0.453333,0.354074,-0.517470,0.139582,0.385965,20.0,23.0,0.0
2,2025_02_BUF_NYJ,2025,2,2025-09-14,REG,NYJ,BUF,0,7,7,...,0.0,0.302612,0.512821,0.457560,-0.046021,0.394435,0.509804,41.0,40.0,1.0
3,2025_02_CAR_ARI,2025,2,2025-09-14,REG,ARI,CAR,0,7,7,...,1.0,-0.176198,0.365079,-0.214952,-0.092878,0.103053,0.460317,10.0,26.0,0.0
4,2025_02_CHI_DET,2025,2,2025-09-14,REG,DET,CHI,0,7,6,...,0.0,-0.053387,0.388060,0.012918,-0.198627,-0.087629,0.425926,24.0,27.0,0.0
